# ABO Adaptive RAG 商品助手
基于 Adaptive-rag 的独立 ABO 项目。运行第一个代码单元格后，可调用 ask()。
支持历史商品取证、规格比较、原图、配件兼容和现有售后政策快照。
日志随问答写入显示的会话目录。待会话结束时可以调用 close_chat()。


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import runpy

if "chat" not in globals():
    script = Path("/kaggle/working/standalone_abo.py")
    script.write_text('"""Start the new Adaptive-rag ABO profile and run the release gates on Kaggle."""\n\nimport atexit\nfrom datetime import datetime, timedelta, timezone\nimport json\nfrom hashlib import file_digest\nfrom pathlib import Path\nimport os\nimport runpy\nfrom shutil import copyfileobj, copytree\nimport subprocess\nimport sys\nfrom uuid import uuid4\nfrom urllib.request import urlopen\nfrom zipfile import ZipFile\n\n\nROOT = Path("/kaggle/working/abo-rag")\nMODEL = Path("/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1")\nOFFICIAL_SOURCE = "https://amazon-berkeley-objects.s3.amazonaws.com/"\n\n\ndef stage_release():\n    inputs = Path("/kaggle/input")\n    markers = list(inputs.rglob("ABO_8B_FORMAL.txt",recurse_symlinks=True))\n    if markers:\n        marker, = markers\n        copytree(marker.parent, ROOT, dirs_exist_ok=True)\n    else:\n        bundle, = inputs.rglob("abo-8b-formal.zip",recurse_symlinks=True)\n        with ZipFile(bundle) as archive:\n            archive.extractall(ROOT)\n    manifest = json.loads((ROOT/"adaptive-release-manifest.json").read_text("utf-8"))\n    for relative, digest in manifest.items():\n        with (ROOT/relative).open("rb") as stream:\n            if file_digest(stream,"sha256").hexdigest() != digest:\n                raise ValueError(f"Release checksum mismatch: {relative}")\n    print(f"Verified {len(manifest)} release files.",flush=True)\n\n\ndef download(url, path):\n    if path.is_file():\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    pending = path.with_suffix(path.suffix+".part")\n    with urlopen(url, timeout=120) as response, pending.open("wb") as output:\n        copyfileobj(response, output)\n    pending.replace(path)\n\n\ndef setup_catalog(workspace=ROOT):\n    source = workspace/"data/ecommerce_rag/abo-source"\n    snapshot = workspace/"offline/abo-source-for-kaggle.zip"\n    if snapshot.is_file():\n        with ZipFile(snapshot) as archive:\n            archive.extractall(workspace)\n    expected = {\n        "archives/abo-listings.tar": "b7f7ceacb328fa5ab6e143b88e1f948443a877cfc95b67ff09c8ebabd50644e3",\n        "images/metadata/images.csv.gz": "915263ef354d7a5d3503e899e610082f510c0011cccd5b4dc4f4c5efa86ef539",\n    }\n    for relative, digest in expected.items():\n        path = source/relative\n        download(OFFICIAL_SOURCE+relative, path)\n        with path.open("rb") as stream:\n            if file_digest(stream,"sha256").hexdigest() != digest:\n                raise ValueError(f"Official ABO input checksum mismatch: {relative}")\n    if not snapshot.is_file():\n        for relative in ("README.md", "LICENSE-CC-BY-4.0.txt"):\n            download(OFFICIAL_SOURCE+relative, source/relative)\n    font = workspace/"fonts/NotoSansSC.ttf"\n    if not font.is_file() and os.name != "nt":\n        base = "https://raw.githubusercontent.com/google/fonts/2894aab31764f10f29c421bdfd2340d3b382d384/ofl/notosanssc/"\n        download(base+"NotoSansSC%5Bwght%5D.ttf", font)\n        download(base+"OFL.txt", font.with_name("OFL.txt"))\n    environment = {**os.environ, "DOC_VQA_WORKSPACE": str(workspace), "ABO_SOURCE_DIR": str(source)}\n    if font.is_file():\n        os.environ["ABO_FONT_PATH"] = str(font)\n    build = workspace/"upstream/VisRAG/personal/ecommerce_rag/full_catalog.py"\n    subprocess.run([sys.executable, str(build)], cwd=workspace, env=environment, check=True)\n\n\ndef configure_graph_runtime():\n    from langchain_core.globals import set_debug, set_verbose\n\n    # Core 0.3 still checks the root LangChain flags when Kaggle preinstalls it.\n    set_debug(False)\n    set_verbose(False)\n\n\ndef start():\n    log = Path("/kaggle/working/ABO启动记录.txt")\n\n    def boot_event(text):\n        with log.open("a",encoding="utf-8") as file:\n            stamp=datetime.now(timezone(timedelta(hours=8))).isoformat(timespec="seconds")\n            file.write(f"[{stamp}] {text}\\n")\n\n    boot_event("开始初始化离线输入、GPU 和 ABO 目录")\n    try:\n        session = _start()\n    except Exception as error:\n        boot_event(f"启动失败 · {type(error).__name__} · 查看 Notebook 执行错误")\n        raise\n    boot_event(f"启动完成 · {session.output}")\n    atexit.register(session.close)\n    return session\n\n\ndef _start():\n    stage_release()\n    if not (MODEL/"config.json").is_file():\n        raise FileNotFoundError(f"Attach the official Kaggle model: {MODEL}")\n    install = [sys.executable,"-m","pip","install","--no-deps"]\n    wheel_dir = ROOT/"offline/wheels"\n    if not wheel_dir.is_dir():\n        subprocess.run([*install,"bitsandbytes==0.50.2","PyMuPDF==2.0","reportlab==5.1.0"],check=True)\n        wheel_dir = ROOT/"graph-wheels"\n    wheels = sorted(wheel_dir.glob("*.whl"))\n    subprocess.run([*install,"--no-index",*map(str,wheels)],check=True)\n    os.environ.update(HF_HUB_OFFLINE="1", HF_HOME="/tmp/abo-adaptive-hf", ABO_QWEN8_MODEL_DIR=str(MODEL),\n                      ABO_QWEN8_REVISION="kaggle:qwen-lm/qwen-3-vl/transformers/8b-instruct/1",\n                      DOC_VQA_WORKSPACE=str(ROOT))\n    setup_catalog()\n    os.chdir(ROOT)\n    sys.path.insert(0,str(ROOT))\n    configure_graph_runtime()\n    from app.abo.service import AboDialogue\n\n    output = ROOT/"reports/ecommerce_rag"/("abo-adaptive-kaggle-"+uuid4().hex[:10])\n    output.mkdir(parents=True)\n    return AboDialogue(output,gpu=True,entry="Kaggle Notebook")\n\n\ndef verify(session):\n    from app.abo.validation import run_category_validation, run_presentation_validation, run_validation\n    from common import WORKSPACE\n\n    checks_script = WORKSPACE/"scripts/kaggle_abo_checks.py"\n    if not checks_script.is_file():\n        checks_script = WORKSPACE/"scripts/kaggle_stage_d_chat.py"\n    smoke = runpy.run_path(str(checks_script))\n    smoke["verify"](session)\n    smoke["verify_categories"](session)\n    result = run_validation(session,session.output/"cross-category.json",require=True)\n    print(f"Cross-category regression {result[\'passed\']}/{result[\'cases\']}.",flush=True)\n    category = run_category_validation(session,session.output/"all-category-followups.json")\n    print(f"Product follow-up across source categories: {category[\'passed\']}/{category[\'categories\']}.",flush=True)\n    presentation = run_presentation_validation(session,session.output/"presentation.json")\n    print(f"Customer reply presentation: {presentation[\'passed\']}/{presentation[\'cases\']}.",flush=True)\n    from common import read_jsonl\n\n    records=read_jsonl(session.output/"results.jsonl")\n    transcript=(session.output/"对话记录.txt").read_text("utf-8")\n    events=read_jsonl(session.output/"运行事件.jsonl")\n    assert {"product","compatibility"} <= {row["entry_lane"] for row in records}\n    assert transcript.count("\\n用户：")==len(records)\n    assert events[0]["event"]=="start" and events[1]["event"]=="ready"\n    print(f"Unified session transcript verified: {len(records)} turns.",flush=True)\n    return result\n', encoding="utf-8")
    bootstrap = runpy.run_path(str(script))
    chat = bootstrap["start"]()

def ask(text, *, details=False):
    row = chat.respond(text)
    print(row["display_answer"])
    for label in dict.fromkeys(source["label"] for source in row["display_sources"]):
        print("来源", label)
    if details:
        return row

def close_chat():
    if "chat" in globals():
        chat.close()
        print("会话日志", chat.output)
        del globals()["chat"]

print("ABO Adaptive RAG 商品助手已就绪")
print("会话日志", chat.output)


In [ ]:
checks = bootstrap["verify"](chat)


In [ ]:
ask("重置");
ask("我想查询电风扇这类商品");


In [ ]:
close_chat()
